# Biochemistry Recurrent Neural Network (LSTM) Example
In this example we will apply an LSTM to the promelt dataset.

# Install Packages
Only run if you do not have packages installed

In [ ]:
%pip install rdkit
%pip install pandas
%pip install scikit-learn
%pip install matplotlib
%pip install torch torchvision torchaudio #note this will install pytorch for CPU

# Set seed and device

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import re
import random
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_SEED = 42

#setting a device allows us to specify to run on cpu or gpu or to let it switch depending on if a gpu is available
#this will run on gpu if available
#because the hyperparameter optimization will be slow, running on gpu will be better
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


#set random seed for all python libraries for preproucibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)

Using device: cpu


# Load data
load data and prepare pytorch datasets

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim

# 20 canonical amino acids
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

# Define vocabulary
vocab = ["PAD"] + list(AMINO_ACIDS)

# Token-to-index and index-to-token mappings
token_to_idx = {token: i for i, token in enumerate(vocab)}
idx_to_token = {i: token for token, i in token_to_idx.items()}

PAD_IDX = token_to_idx["PAD"]

def tokenize_protein(sequence):
    """Convert a protein sequence to a list of integer token IDs."""
    return [token_to_idx[aa] for aa in sequence]


def encode_proteins(df, sequence_col="sequence", max_length=None):

    df = df.copy()

    if max_length is None:
        max_length = df[sequence_col].str.len().max()

    df = df[df[sequence_col].str.len() <= max_length].copy()

    lengths = df[sequence_col].str.len().to_numpy()

    encoded = np.full(
        (len(df), max_length),
        PAD_IDX,
        dtype=np.int64
    )

    for i, sequence in enumerate(df[sequence_col]):
        tokens = tokenize_protein(sequence)
        encoded[i, :len(tokens)] = tokens

    return encoded, lengths, df

df = pd.read_csv("full_promelt_seq.csv")
#remove rows with invalid amino acids
df = df[df["sequence"].str.count("X")==0]
df = df[df["sequence"].str.count("U")==0]
df = df[df["sequence"].str.count("Z")==0]
df = df[df["sequence"].str.count("B")==0]

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42)

#encode protein sequences
train_encoded, train_lengths, train_df = encode_proteins(train_df, max_length=800)
val_encoded, val_lengths, val_df = encode_proteins(val_df, max_length=800)

#make tensors and datasets
X_train = torch.tensor(train_encoded, dtype=torch.long)
X_val = torch.tensor(val_encoded, dtype=torch.long)
y_train = torch.tensor(train_df["Tm"].values.astype(np.float32))
y_val = torch.tensor(val_df["Tm"].values.astype(np.float32))
train_lengths = torch.tensor(train_lengths,dtype=torch.long)

val_lengths = torch.tensor(val_lengths, dtype=torch.long)

train_dataset = TensorDataset(X_train, train_lengths, y_train)
val_dataset = TensorDataset(X_val, val_lengths, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Train model

In [ ]:
#class for an LSTM regression model
class ProteinLSTMRegressor(nn.Module):
    def __init__(
        self,
        vocab_size=21, #20 aa + 1 PAD character
        embedding_dim=64,
        hidden_dim=128,
        num_layers=1,
        dropout=0.2,
        pad_idx=0,
        bidirectional=False
    ):
        super().__init__()
        
        #turns input characters into a learned embedding, or representation, for each character
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        
        #set up the LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
    
        #create output layer
        self.regressor = nn.Sequential(
            nn.Linear(lstm_output_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    #forward method predicts the output
    def forward(self, x, lengths):
        # x shape: (batch, seq_len)
        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (h_n, c_n) = self.lstm(packed)

        # h_n shape:
        #   unidirectional: (num_layers, batch, hidden_dim)
        #   bidirectional: (num_layers*2, batch, hidden_dim)
        if self.lstm.bidirectional:
            # concatenate final forward + backward hidden states
            final_hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            final_hidden = h_n[-1]

        out = self.regressor(final_hidden)
        return out
    
#create the model, you can try adjusting hyperparameters like the embedding and hidden dimensions
#you can also try adding multiple layers which will run the output of the one LSTM through a second LSTM
model = ProteinLSTMRegressor(
    embedding_dim=64,
    hidden_dim=128,
    num_layers=1,
    dropout=0.2,
    pad_idx=PAD_IDX,
    bidirectional=True
).to(DEVICE)

criterion = nn.MSELoss() #MSELoss for regression
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(model, train_loader, val_loader, criterion, optimizer): 
    total_loss = 0.0
    train_preds = []
    train_true = []
    
    model.train()
    for x, lengths, y in train_loader:
        x = x.to(DEVICE)
        lengths = lengths.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        preds = model(x, lengths)
        loss = criterion(preds.squeeze(), y)

        
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        train_preds.extend(preds.cpu().detach().numpy())
        train_true.extend(y.cpu().numpy())

    
    train_rmse = np.sqrt(mean_squared_error(train_true, train_preds))
    train_r2 = r2_score(train_true, train_preds)
    train_avg_loss = total_loss / len(train_loader.dataset)
    
    val_preds = []
    val_true = []
    model.eval()
    with torch.no_grad():
        for x, lengths, y in val_loader:
            x = x.to(DEVICE)
            lengths = lengths.to(DEVICE)
            y = y.to(DEVICE)
            
            preds = model(x, lengths)
            val_preds.extend(preds.cpu().detach().numpy())
            val_true.extend(y.cpu().numpy())
    val_rmse = np.sqrt(mean_squared_error(val_true, val_preds))
    val_r2 = r2_score(val_true, val_preds)
    
    return train_avg_loss, train_rmse, train_r2, val_rmse, val_r2


#train model
num_epochs = 15

for epoch in range(1, num_epochs + 1):
    train_loss, train_rmse, train_r2, val_rmse, val_r2 = run_epoch(model, train_loader, val_loader, criterion, optimizer)
    

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train RMSE: {train_rmse:.4f} | Train R2: {train_r2:.4f} | "
        f"Val RMSE: {val_rmse:.4f} | Val R2: {val_r2:.4f}"
    )

# Validation set evaluation

In [ ]:
import matplotlib.pyplot as plt

#get the actual predictions
all_preds = []
all_true = []
model.eval()
for x, lengths, y in val_loader:
    x = x.to(DEVICE)
    lengths = lengths.to(DEVICE)
    y = y.to(DEVICE)
    with torch.no_grad():
        preds = model(x, lengths)
    
    all_preds.extend(preds.cpu().detach().numpy())
    all_true.extend(y.cpu().numpy())

plt.figure(figsize=(6, 6))
plt.scatter(all_true, all_preds, alpha=0.7)

min_val = min(min(all_true), min(all_preds))
max_val = max(max(all_true), max(all_preds))
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.title(f"Val True vs Predicted")
plt.tight_layout()
plt.show()